# Week 5: Baseline Individual Classifiers
**Author:** Vishnu  
**Goal:** Train simple classifiers with default parameters and record performance.

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os
project_path = '/content/drive/MyDrive/ParkinsonsProject/ParkinsonsDetection'
os.chdir(project_path)

import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

Mounted at /content/drive


## Load Feature‑Selected Data

In [2]:
X_train = np.load('data/processed/X_train_fs.npy')
y_train = np.load('data/processed/y_train.npy')
X_test = np.load('data/processed/X_test_fs.npy')
y_test = np.load('data/processed/y_test.npy')

## Define Classifiers
We'll use default hyperparameters for baseline.

In [3]:
classifiers = {
    'Random Forest': RandomForestClassifier(random_state=42),
    'SVM': SVC(kernel='rbf', probability=True, random_state=42),
    'k‑NN': KNeighborsClassifier(),
    'Logistic Regression': LogisticRegression(random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42)
}

## Train and Evaluate
We'll use 5‑fold cross‑validation on training set and then test set performance.

In [4]:
results = []
for name, clf in classifiers.items():
    # Cross‑validation AUC
    cv_scores = cross_val_score(clf, X_train, y_train, cv=5, scoring='roc_auc')
    clf.fit(X_train, y_train)
    # Test predictions
    y_pred = clf.predict(X_test)
    y_proba = clf.predict_proba(X_test)[:,1]
    results.append({
        'Classifier': name,
        'CV AUC (mean)': cv_scores.mean(),
        'CV AUC (std)': cv_scores.std(),
        'Test Accuracy': accuracy_score(y_test, y_pred),
        'Test Precision': precision_score(y_test, y_pred),
        'Test Recall': recall_score(y_test, y_pred),
        'Test F1': f1_score(y_test, y_pred),
        'Test AUC': roc_auc_score(y_test, y_proba)
    })
    # Save model
    safe_name = name.lower().replace(' ', '_').replace('-', '').replace('‑', '')
    joblib.dump(clf, f'models/{safe_name}_baseline.joblib')
    print(f"Saved {safe_name}_baseline.joblib")

Saved random_forest_baseline.joblib
Saved svm_baseline.joblib
Saved knn_baseline.joblib
Saved logistic_regression_baseline.joblib
Saved decision_tree_baseline.joblib


## Display Results

In [5]:
results_df = pd.DataFrame(results)
print(results_df.to_string())
# Save to CSV
results_df.to_csv('reports/baseline_results.csv', index=False)

            Classifier  CV AUC (mean)  CV AUC (std)  Test Accuracy  Test Precision  Test Recall   Test F1  Test AUC
0        Random Forest       0.988421      0.021866       0.803279        0.803279     1.000000  0.890909  0.716837
1                  SVM       0.977895      0.042907       0.688525        0.777778     0.857143  0.815534  0.467687
2                 k‑NN       0.970013      0.029001       0.737705        0.836735     0.836735  0.836735  0.634354
3  Logistic Regression       0.933789      0.103974       0.704918        0.792453     0.857143  0.823529  0.675170
4        Decision Tree       0.887895      0.112197       0.672131        0.784314     0.816327  0.800000  0.449830


## Identify Best Base Learners
Look at Test AUC or F1. Choose 3‑4 models with highest performance to use in stacking.

In [6]:
results_df.sort_values('Test AUC', ascending=False)

,Classifier,CV AUC (mean),CV AUC (std),Test Accuracy,Test Precision,Test Recall,Test F1,Test AUC
0,Random Forest,0.988421,0.021866,0.803279,0.803279,1.000000,0.890909,0.716837
3,Logistic Regression,0.933789,0.103974,0.704918,0.792453,0.857143,0.823529,0.675170
2,k‑NN,0.970013,0.029001,0.737705,0.836735,0.836735,0.836735,0.634354
1,SVM,0.977895,0.042907,0.688525,0.777778,0.857143,0.815534,0.467687
4,Decision Tree,0.887895,0.112197,0.672131,0.784314,0.816327,0.800000,0.449830


## Done
Baseline models saved. Proceed to Week 6 (Stacking Ensemble).